In [0]:
from pyspark.sql import SparkSession

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("HighestSalaryPerGenderSQL").getOrCreate()

# 2. Create the Input Data and register it as a temporary view
data = [
    (1, "sara", "2k", "Female"),
    (2, "joh", "3k", "Female"),
    (3, "jeo", "5k", "Male"),
    (4, "cemo", "6k", "Male"),
    (5, "triv", "2k", "Female"),
    (6, "buch", "5k", "Male"),
    (7, "rev", "10k", "Female")
]

df = spark.createDataFrame(data, ["ID", "NAME", "SAL", "GENDER"])
df.createOrReplaceTempView("input_table")

print("--- Original Input Table ---")
spark.sql("SELECT * FROM input_table").show()

# 3. Spark SQL Query using FIRST_VALUE and windowing
df_new = spark.sql("""
    SELECT 
        ID, 
        NAME, 
        SAL, 
        GENDER AS GENDER_cat,
        
        -- Grabs the name of the highest earner per gender partition
        FIRST_VALUE(NAME) OVER (
            PARTITION BY GENDER 
            ORDER BY CAST(REPLACE(SAL, 'k', '') AS INT) DESC
        ) AS Higest_sal_name
        
    FROM input_table
    ORDER BY GENDER, ID
""")

display(df_new)

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("PerformanceTransformation").getOrCreate()

# Sample input data matching your interview question
data = [
    ("G2 Outperformed / L1 Role Model",),
    ("G2 highperformed / L3 consistence performance",)
]

# Create DataFrame
df = spark.createDataFrame(data, ["perf_id"])

# Register as a temporary view so we can use Spark SQL
df.createOrReplaceTempView("input_table")

df_new = spark.sql("""
    SELECT 
        -- 1. Construct perf_id
        CONCAT(
            REGEXP_SUBSTR(perf_id, 'G[0-9]+'), 
            REGEXP_SUBSTR(perf_id, 'L[0-9]+')
        ) AS perf_id,
        
        -- 2. Calculate perf_value
        CAST(REGEXP_REPLACE(REGEXP_SUBSTR(perf_id, 'G[0-9]+'), '[^0-9]', '') AS INT) + 
        CAST(REGEXP_REPLACE(REGEXP_SUBSTR(perf_id, 'L[0-9]+'), '[^0-9]', '') AS INT) AS perf_value

    FROM input_table
""")

display(df_new)